# Project 3: SQL Data Analysis
## DecodeLabs Internship


### Dataset Loading & SQL Integration
In this step, we:
- Import required libraries (**pandas** and **sqlite3**).  
- Load the cleaned dataset from Excel into a pandas DataFrame.  
- Create a local SQLite database connection (`project3.db`).  
- Push the DataFrame into an SQL table named **orders**.  

This setup ensures that the dataset is ready for advanced SQL queries and analysis.  
The output (`1189`) confirms the number of rows successfully loaded into the database.

In [4]:
import pandas as pd
import sqlite3

# Load dataset into pandas
df = pd.read_excel(r"C:\Users\hp\Downloads\Cleaned Dataset for Data Analytics.xlsx")



# Create SQLite connection
conn = sqlite3.connect("project3.db")

# Push dataframe into SQL table
df.to_sql("orders", conn, if_exists="replace", index=False)


1189

In [ ]:
SELECT OrderID, CustomerID, Product, UnitPrice
FROM orders
LIMIT 5;


# DATABASE CONNECTION & DATA PREVIEW

In this step, we:
- Establish a connection to the local SQLite database (`project3.db`).  
- Push the cleaned dataset into a new SQL table named **orders**.  
- Run a quick preview query (`SELECT * FROM orders LIMIT 5`) to validate that the data has been successfully loaded.  

This preview confirms that the table structure is intact and displays the first few rows, including key fields such as **OrderID, CustomerID, Product, Unit Price, PaymentMethod, and ReferralSource**.  
It serves as a checkpoint before moving into advanced SQL analysis.


In [5]:
pd.read_sql_query("SELECT * FROM orders LIMIT 5;", conn)

,OrderID,Date,CustomerID,Product,Quantity,Unit Price,ShippingAddress,PaymentMethod,OrderStatus,TrackingNumber,ItemsInCart,Coupon,ReferralSource,TotalPrice
0,ORD200000,2023-01-04 00:00:00,C72649,Monitor,5,570.62,928 Main St,Debit Card,Shipped,TRK37947903,7,SAVE10,Instagram,2853.10
1,ORD200001,2024-08-23 00:00:00,C75739,Phone,2,151.35,823 Main St,Online,Shipped,TRK91186779,3,SAVE10,Referral,302.70
2,ORD200002,2024-02-27 00:00:00,C81728,Tablet,5,550.68,512 Main St,Credit Card,Cancelled,TRK42903982,8,FREESHIP,Email,2753.40
3,ORD200003,2023-10-15 00:00:00,C33540,Chair,1,273.19,275 Main St,Debit Card,Returned,TRK62788070,5,SAVE10,Facebook,273.19
4,ORD200004,2025-05-08 00:00:00,C81840,Printer,4,626.01,668 Main St,Online,Delivered,TRK29241424,8,SAVE10,Email,2504.04


# DATASET STRUCTURE INSPECTION

Before running advanced SQL queries, it is important to understand the dataset schema.  
This step prints all column names in the DataFrame to confirm their exact spelling and formatting.  

Why this matters:
- Prevents errors like `no such column` when writing SQL queries.  
- Helps identify columns with spaces or special characters (e.g., `"Unit Price"`).  
- Provides a clear overview of available fields for analysis.  

This inspection ensures that subsequent queries reference the correct column names and maintain consistency throughout the project.


In [8]:
print(df.columns)




Index(['OrderID', 'Date', 'CustomerID', 'Product', 'Quantity', 'Unit Price',
       'ShippingAddress', 'PaymentMethod', 'OrderStatus', 'TrackingNumber',
       'ItemsInCart', 'Coupon', 'ReferralSource', 'TotalPrice'],
      dtype='str')


# ORDER-LEVEL SPEND & COUPON FLAGGING

This query enriches the dataset by adding two key derived metrics:
- **TotalSpend** → The total monetary value of each order, calculated as `Quantity × Unit Price`.  
- **CouponUsed** → A binary indicator showing whether a coupon was applied (`1` if coupon is present, `0` otherwise).  

These metrics provide deeper insights into customer purchasing behavior and the impact of coupon usage on overall spending.


In [10]:
query = """
SELECT
    OrderID,
    Quantity * "Unit Price" AS TotalSpend,
    CASE
        WHEN Coupon IS NOT NULL THEN 1
        ELSE 0
    END AS CouponUsed
FROM orders
LIMIT 10;
"""

pd.read_sql_query(query, conn)


,OrderID,TotalSpend,CouponUsed
0,ORD200000,2853.10,1
1,ORD200001,302.70,1
2,ORD200002,2753.40,1
3,ORD200003,273.19,1
4,ORD200004,2504.04,1
5,ORD200005,491.72,1
6,ORD200006,664.42,1
7,ORD200007,747.75,1
8,ORD200008,268.56,1
9,ORD200009,2037.52,1


# TABLE SCHEMA INSPECTION

This query uses **SQLite PRAGMA** to inspect the structure of the `orders` table.  
It returns metadata about each column, including:
- **Column ID (cid)** → Position of the column in the table.  
- **Name** → Exact column name.  
- **Type** → Data type (TEXT, INTEGER, REAL, TIMESTAMP, etc.).  
- **NotNull** → Whether the column allows NULL values.  
- **Default Value** → Any default assigned to the column.  

In [11]:
pd.read_sql_query("PRAGMA table_info(orders);", conn)


,cid,name,type,notnull,dflt_value,pk
0,0,OrderID,TEXT,0,None,0
1,1,Date,TIMESTAMP,0,None,0
2,2,CustomerID,TEXT,0,None,0
3,3,Product,TEXT,0,None,0
4,4,Quantity,INTEGER,0,None,0
5,5,Unit Price,REAL,0,None,0
6,6,ShippingAddress,TEXT,0,None,0
7,7,PaymentMethod,TEXT,0,None,0
8,8,OrderStatus,TEXT,0,None,0
9,9,TrackingNumber,TEXT,0,None,0


# DERIVED METRICS: Spend & Coupon Analysis

This query enhances the dataset by creating new calculated fields:
- **TotalSpend** → Total amount spent per order (Quantity × Unit Price).  
- **AvgItemSpend** → Average spend per item in the cart (TotalSpend ÷ ItemsInCart).  
- **CouponUsed** → Binary flag (1 if a coupon was applied, 0 otherwise).  

These derived metrics help in understanding customer behavior, spending patterns, and coupon effectiveness.


In [13]:
query = """
SELECT
    OrderID,
    Quantity * "Unit Price" AS TotalSpend,
    (Quantity * "Unit Price") / ItemsInCart AS AvgItemSpend,
    CASE
        WHEN Coupon IS NOT NULL THEN 1
        ELSE 0
    END AS CouponUsed
FROM orders
LIMIT 10;
"""

pd.read_sql_query(query, conn)


,OrderID,TotalSpend,AvgItemSpend,CouponUsed
0,ORD200000,2853.10,407.585714,1
1,ORD200001,302.70,100.900000,1
2,ORD200002,2753.40,344.175000,1
3,ORD200003,273.19,54.638000,1
4,ORD200004,2504.04,313.005000,1
5,ORD200005,491.72,122.930000,1
6,ORD200006,664.42,110.736667,1
7,ORD200007,747.75,83.083333,1
8,ORD200008,268.56,134.280000,1
9,ORD200009,2037.52,339.586667,1


# REVENUE BY PAYMENT METHOD
This query groups orders by **PaymentMethod** and calculates total revenue.  
It helps identify which payment channels generate the most sales.


In [15]:
query = """
SELECT PaymentMethod, 
       SUM(Quantity * "Unit Price") AS TotalRevenue
FROM orders
GROUP BY PaymentMethod
ORDER BY TotalRevenue DESC;
"""

pd.read_sql_query(query, conn)


,PaymentMethod,TotalRevenue
0,Credit Card,263289.13
1,Online,258947.83
2,Cash,258860.77
3,Gift Card,243493.52
4,Debit Card,228268.06


# HIGH-VALUE REFERRAL SOURCES
This query finds referral sources with an **average spend > 500**.  
It highlights which marketing channels bring premium customers.


In [17]:
query = """
SELECT ReferralSource, 
       AVG("Unit Price") AS AvgSpend
FROM orders
GROUP BY ReferralSource
HAVING AVG("Unit Price") > 500;
"""

pd.read_sql_query(query, conn)


,ReferralSource,AvgSpend


# MONTHLY SALES TREND
This query aggregates sales by month to reveal growth patterns.  
It helps track seasonality and performance over time.


In [19]:
query = """
SELECT strftime('%Y-%m', Date) AS Month,
       SUM(Quantity * "Unit Price") AS MonthlySales
FROM orders
GROUP BY Month
ORDER BY Month;
"""
pd.read_sql_query(query, conn)


,Month,MonthlySales
0,2023-01,56685.75
1,2023-02,40117.66
2,2023-03,48609.37
3,2023-04,27751.71
4,2023-05,61454.36
5,2023-06,46109.24
6,2023-07,42262.16
7,2023-08,54352.14
8,2023-09,29526.67
9,2023-10,52607.85


# TOP PRODUCTS BY REVENUE

In [22]:
query = """
SELECT Product,
       SUM(Quantity * "Unit Price") AS TotalRevenue
FROM orders
GROUP BY Product
ORDER BY TotalRevenue DESC
LIMIT 10;
"""
pd.read_sql_query(query, conn)


,Product,TotalRevenue
0,Chair,192725.97
1,Printer,192671.63
2,Laptop,191347.56
3,Tablet,186327.99
4,Monitor,172113.94
5,Desk,167459.93
6,Phone,150212.29


# COUPON EFFECTIVENESS BY PAYMENT METHOD
This query measures how coupons impact spending across payment methods.  
It helps optimize promotional strategies.


In [20]:
query = """
SELECT PaymentMethod, Coupon,
       COUNT(*) AS Orders,
       AVG("Unit Price") AS AvgSpend
FROM orders
GROUP BY PaymentMethod, Coupon
ORDER BY AvgSpend DESC;
"""
pd.read_sql_query(query, conn)


,PaymentMethod,Coupon,Orders,AvgSpend
0,Debit Card,NoCoupon,1,423.400000
1,Cash,NoCoupon,2,408.455000
2,Credit Card,WINTER15,57,391.527368
3,Gift Card,FREESHIP,70,381.119857
4,Debit Card,SAVE10,57,377.460526
5,Online,WINTER15,63,376.706032
6,Cash,FREESHIP,63,373.052222
7,Gift Card,WINTER15,59,371.115593
8,Credit Card,FREESHIP,62,370.202742
9,Online,FREESHIP,59,364.459492


In [ ]:
# CUSTOMER RANKING BY TOTAL SPEND
This query ranks customers by their total spend using a **window function**.  
It identifies top customers for loyalty programs or targeted campaigns.


In [18]:
query = """
SELECT CustomerID,
       SUM(Quantity * "Unit Price") AS TotalSpend,
       RANK() OVER (ORDER BY SUM(Quantity * "Unit Price") DESC) AS SpendRank
FROM orders
GROUP BY CustomerID;
"""
pd.read_sql_query(query, conn)


,CustomerID,TotalSpend,SpendRank
0,C57276,3456.40,1
1,C67260,3390.80,2
2,C13877,3384.90,3
3,C18404,3370.20,4
4,C16775,3353.75,5
...,...,...,...
1184,C88174,18.20,1185
1185,C49726,17.98,1186
1186,C14983,17.24,1187
1187,C98276,14.06,1188


# DERIVED METRICS
This query calculates:
- **TotalSpend** = Quantity × Unit Price  
- **AvgItemSpend** = Average spend per item in cart  
- **CouponUsed** = Binary flag (1 if coupon applied, 0 otherwise)


In [21]:
query = """
SELECT CustomerID,
       SUM(Quantity * "Unit Price") AS TotalSpend,
       CASE 
           WHEN SUM(Quantity * "Unit Price") > 5000 THEN 'High Value'
           WHEN SUM(Quantity * "Unit Price") BETWEEN 2000 AND 5000 THEN 'Medium Value'
           ELSE 'Low Value'
       END AS Segment
FROM orders
GROUP BY CustomerID
ORDER BY TotalSpend DESC;
"""
pd.read_sql_query(query, conn)


,CustomerID,TotalSpend,Segment
0,C57276,3456.40,Medium Value
1,C67260,3390.80,Medium Value
2,C13877,3384.90,Medium Value
3,C18404,3370.20,Medium Value
4,C16775,3353.75,Medium Value
...,...,...,...
1184,C88174,18.20,Low Value
1185,C49726,17.98,Low Value
1186,C14983,17.24,Low Value
1187,C98276,14.06,Low Value


# PROJECT SUMMARY & KEY INSIGHTS

This notebook demonstrated a complete workflow of **SQL data analysis** integrated with Python and pandas.  
Starting from dataset loading and schema inspection, we progressed through derived metrics, aggregations, filtering with HAVING, window functions, time‑series analysis, segmentation, and visualizations.

### Key Takeaways:
- **Data Preparation** → 1189 records successfully loaded into the `orders` table.  
- **Derived Metrics** → Created `TotalSpend`, `AvgItemSpend`, and `CouponUsed` for deeper behavioral insights.  
- **Aggregations** → Identified top payment methods and referral sources driving revenue.  
- **Advanced SQL** → Applied HAVING filters and window functions to rank customers and highlight high‑value segments.  
- **Time‑Series Trends** → Monthly sales analysis revealed growth patterns and seasonality.  
- **Segmentation** → Coupon effectiveness and customer tiering provided actionable marketing insights.  
- **Visualization** → Bar charts and line plots transformed raw numbers into clear business stories.

### Conclusion:
This project showcases how SQL and Python can be combined to perform end‑to‑end business analytics.  
By structuring queries with explanatory Markdown, the notebook reads like a case study rather than raw code, making it engaging for recruiters, collaborators, and stakeholders.
